# Café Data Analysis
On this notebook we will be analysing the data collected by a Café over a period of a year.

Our task is to explore the data, identify intersting patterns and develop insights to help the business improve its performance.

In [5]:
# Libraries
import sys
!{sys.executable} -m pip install numpy pandas seaborn matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load file


In [6]:
# Variables

In [7]:
# Import and inspect
df = pd.read_csv('cafe.csv', delimiter=';', encoding='latin-1') # Had to add encoding, donno why tho
df.head()

,Date,Bill Number,Item Desc,Time,Quantity,Rate,Tax,Discount,Total,Category
0,1-Apr-10,G0470115,QUA MINERAL WATER(1000ML),13:15:11,1,50,"11,88",0,"61,88",BEVERAGE
1,1-Apr-10,G0470115,MONSOON MALABAR (AULAIT),13:15:11,1,100,"23,75",0,"123,75",BEVERAGE
2,1-Apr-10,G0470116,MASALA CHAI CUTTING,13:17:35,1,40,"9,5",0,"49,5",BEVERAGE
3,1-Apr-10,G0470117,QUA MINERAL WATER(1000ML),13:19:55,1,50,"11,88",0,"61,88",BEVERAGE
4,1-Apr-10,G0470283,MOROCCAN MINT TEA,01:20:18,1,45,"10,69",0,"55,69",BEVERAGE


Questions telmo would like to get answered:
1 - what are the most sold products in total
2 - what is the income in total
3 - what are the most sold products per month
4 - what are the most sold products per week (including day / histogram)
5 - what products are usually bought together (see per day, per week, per month)
6 - what are the most sold products per quartile (each 3 months or per season)
7 - Do products with discount end up giving more profit?
8 - 

In [8]:
df.shape

(145830, 10)

The dataset contains 145_830 rows and 10 columns.

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145830 entries, 0 to 145829
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   Date          145830 non-null  str  
 1   Bill Number   145830 non-null  str  
 2   Item Desc     145830 non-null  str  
 3   Time          145830 non-null  str  
 4   Quantity      145830 non-null  int64
 5   Rate          145830 non-null  str  
 6   Tax           145830 non-null  str  
 7   Discount      145830 non-null  str  
 8   Total         145830 non-null  str  
 9   Category      145830 non-null  str  
dtypes: int64(1), str(9)
memory usage: 11.1 MB


Here I noticed that the rate, tax, discount and total were strings instead of floats (because it's money) and that the time is also string and not Datetime.

Another thing is that there is a column called category which can be used to categorize the data.

In [10]:
df.nunique()

Date              365
Bill Number     69982
Item Desc         580
Time            36200
Quantity           20
Rate              134
Tax               445
Discount          111
Total             470
Category            9
dtype: int64

This shows us the 365 dates (days) and 9 different categories.

In [11]:
df.describe()

,Quantity
count,145830.000000
mean,1.121299
std,0.477237
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,30.000000


Describe only shows Quantity, since like mentioned before, columns that should be floats, aren't. This step will be repeated later.

This next step will transform those columns into the right type.

The columns that must have it's type changed aren't in the good format, the numbers have a ',' instead of '.' which is how python read floats. This will all be tackled in the next step.

In [12]:
float_cols = ["Rate", "Tax", "Discount", "Total"]
for col in float_cols:
    df[col] = pd.to_numeric(
        df[col].str.replace(',', '.', regex=False),
        errors='coerce'
    )

df["Date"] = df["Date"].astype('datetime64[ns]')


In [13]:
df.describe()

,Date,Quantity,Rate,Tax,Discount,Total
count,145830,145830.000000,145830.000000,145830.000000,145830.000000,145830.000000
mean,2010-10-04 15:52:45.126516992,1.121299,161.782259,48.929061,0.095079,224.959852
min,2010-04-01 00:00:00,1.000000,0.010000,0.000000,0.000000,0.010000
25%,2010-07-10 00:00:00,1.000000,95.000000,22.560000,0.000000,117.560000
50%,2010-10-07 00:00:00,1.000000,125.000000,32.060000,0.000000,167.060000
75%,2011-01-01 00:00:00,1.000000,225.000000,72.000000,0.000000,315.000000
max,2011-03-31 00:00:00,30.000000,2100.000000,2731.250000,825.000000,14231.250000
std,NaN,0.477237,102.244631,40.272851,3.720735,164.960776


Something interesting that popped to me immediately is how the min for Rate (how much something costs before tax) is 0.01 cents and the max is 2100 euros, this seems highly suspicious and flagged for later exploration, but for now let's continue exploring

## Missing values

Next we decided to see if there are any missing values and think on how to handle them

In [14]:
df.isnull().sum()

Date            0
Bill Number     0
Item Desc       0
Time            0
Quantity        0
Rate            0
Tax             0
Discount        0
Total           0
Category        0
dtype: int64

The dataset doesn't contain any missing values, this step can be safely skipped.

## Visualization of the raw data
In the words of my professor: "Humans are visuals beings so make visualizations", this is what this next step is about.